# You cannot test an estimator against data whose answer you do not know

Every recovery test in axiom needs a world where the true effect is a number somebody wrote
down, not a number the estimator produced. Real data cannot do that job: on real data, an
estimator that is wrong by 30% and an estimator that is right look the same.

`LinearSCM` composes a `CausalGraph` with edge coefficients, noise scales, latent scales for
bidirected edges, and intercepts; it simulates observational and interventional data and
knows its own truth. The named worlds below each exist because some estimator in this
repository claims to handle one specific failure, and each world is that failure, isolated.

In [ ]:
import numpy as np

from axiom.core import Spec
from axiom.identify import ols
from axiom.sim import (
    LinearSCM, SCMError, coefficient_key, confounded_world, feedback_world, frontdoor_world,
    hidden_confounder_world, iv_world, latent_key, mediator_world, transport_pair,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, intervals

enable();  # every axiom result renders itself from here on

In [ ]:
scm = LinearSCM.from_text("Z -> X: 0.8, Z -> Y: 1.5, X -> Y: 2.0, X <-> Y: 0.7", name="demo")
print(scm.graph)
print(scm.coefficients, scm.latent_sd)
print(coefficient_key("Z", "X"), latent_key("Y", "X"))
print("total effect X -> Y:", scm.total_effect("X", "Y"), "| direct:", scm.direct_effect("X", "Y"))

In [ ]:
frame = scm.simulate(5, seed=0)
print(frame.round(3))
do1 = scm.simulate(100_000, seed=0, intervene={"X": 1.0})
do0 = scm.simulate(100_000, seed=0, intervene={"X": 0.0})
print("E[Y|do(1)] - E[Y|do(0)] =", round(float(do1["Y"].mean() - do0["Y"].mean()), 3))
print("exact:", scm.interventional_mean("Y", intervene={"X": 1.0}) - scm.interventional_mean("Y", intervene={"X": 0.0}))

Two routes to the same number: `intervene=` runs the surgery on the simulator, and
`interventional_mean` is the closed form. They agree, which is what makes the simulator
trustworthy as a reference rather than merely convenient.

## Named worlds

Each documents its truth and what a naive estimator gets wrong.

In [ ]:
rows = []
for w in (confounded_world, hidden_confounder_world, iv_world, frontdoor_world, mediator_world, feedback_world):
    s = w()
    rows.append(
        [w.__name__, s.graph.to_text(), f"{s.total_effect('X', 'Y'):.2f}",
         str(s.graph.unmeasured), str(s.graph.feedback)]
    )
table(rows, headers=("world", "graph", "true effect", "unmeasured", "feedback"))
src, tgt = transport_pair()
print("transport pair differs in Z:", src.intercepts, "->", tgt.intercepts, "| selection:", tgt.graph.selection)

In [ ]:
naive = []
for w in (confounded_world, hidden_confounder_world, iv_world, frontdoor_world, mediator_world):
    world = w()
    observed = world.observed(world.simulate(20_000, seed=0))
    est = ols(observed, "Y", "X")
    tr = world.total_effect("X", "Y")
    naive.append((w.__name__.replace("_world", ""), est.estimate / tr,
                  (est.estimate - 2 * est.se) / tr, (est.estimate + 2 * est.se) / tr))

fig = intervals(
    naive, ref=1.0, ref_label="the truth",
    highlight="mediator",
    title="What each world is for",
    subtitle="the obvious regression of Y on X, as a fraction of the effect that actually exists",
    x_title="naive estimate ÷ truth",
)
caption(fig, "Four rows miss by 17% to 36%, and each has an estimator in axiom.identify "
             "that fixes exactly that one. The highlighted row is the opposite test: "
             "regressing on X alone already recovers the total effect through a mediator, so "
             "a package that 'corrects' it — by adjusting for M — has broken something that "
             "worked.")

## Validation and identity

Coefficients must match the graph's edges exactly; the SCM is a `Spec`, so a world is a
hashable, serializable artifact a test can pin. A recovery test that drifted because somebody
retuned the simulator would be worse than no test at all.

In [ ]:
try:
    LinearSCM.from_text("X -> Y: 1.0, X -> Y: 2.0")
except SCMError as e:
    print("refused:", e)
print(Spec.from_json(scm.to_json()) == scm, scm.content_hash()[:16])
print(scm.observed(frame).columns.tolist(), "| unmeasured dropped:", hidden_confounder_world().observed(hidden_confounder_world().simulate(3, seed=0)).columns.tolist())

Note the last line: `observed()` drops the unmeasured columns. A simulator that handed the
estimator the latent variable it is supposed to be robust to would pass every test and prove
nothing.

## What this bought you

Six standard failure modes as objects with known answers, hashable so a test pins one
exactly, and a simulator that agrees with its own closed form. That is the substrate every
`@pytest.mark.recovery` test in the repository stands on.